In [ ]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect('Pokemon')
consulta = "SELECT * FROM batallas ORDER BY id DESC"
tabla_historial = pd.read_sql_query(consulta, conexion)
conexion.close()
display(tabla_historial)

,id,fecha,jugador_1,jugador_2,ganador,numero_turnos
0,1,2026-11-23 22:34:40,Jugador 1,Jugador 2,Jugador 2,4


In [ ]:
import sqlite3
import random
import os
import base64
import ipywidgets as widgets
from IPython.display import display

class RepositorioPokemon:
  def __init__(self, ruta_base_datos='Pokemon'):
    self.ruta_base_datos = ruta_base_datos

  def crear_conexion(self):
    conexion = sqlite3.connect(self.ruta_base_datos)
    conexion.row_factory = sqlite3.Row
    conexion.execute("PRAGMA foreign_keys = ON")
    return conexion

  def listar_pokemones(self):
    with self.crear_conexion() as conexion:
      cursor = conexion.cursor()
      cursor.execute("SELECT ID, Nombre FROM Pokemones")
      return [dict(fila) for fila in cursor.fetchall()]

  def cargar_pokemon(self, id_pokemon):
    with self.crear_conexion() as conexion:
      cursor = conexion.cursor()
      cursor.execute("SELECT * FROM Pokemones WHERE ID = ?", (id_pokemon,))
      fila_pokemon = cursor.fetchone()
      if not fila_pokemon: return None
      cursor.execute("SELECT m.* FROM Movimientos m JOIN Pokemon_Movimientos pm ON m.ID = pm.ID_Movimiento WHERE pm.ID_Pokemon = ?", (id_pokemon,))
      filas_movimientos = cursor.fetchall()
      cursor.execute("SELECT me.ID_Movimiento, e.Nombre, COALESCE(me.Probabilidad, e.Probabilidad) as Prob FROM Movimiento_Efectos me JOIN Efectos e ON me.ID_Efecto = e.ID JOIN Pokemon_Movimientos pm ON me.ID_Movimiento = pm.ID_Movimiento WHERE pm.ID_Pokemon = ?", (id_pokemon,))
      efectos_por_movimiento = {}
      for ef in cursor.fetchall(): efectos_por_movimiento.setdefault(ef['ID_Movimiento'], []).append(dict(ef))

      lista_movimientos = []
      for f in filas_movimientos:
        mov = Movimiento(f['Nombre'], f['ID_Tipo'], f['Categoria'], f['Potencia'] or 0, f['Precision'] or 100, f['PP'] or 10)
        mov.lista_efectos = efectos_por_movimiento.get(f['ID'], [])
        lista_movimientos.append(mov)

      pokemon = Pokemon(fila_pokemon['ID'], fila_pokemon['Nombre'], fila_pokemon['ID_Tipo1'], fila_pokemon['ID_Tipo2'], fila_pokemon['PS'], fila_pokemon['Ataque'], fila_pokemon['Defensa'], fila_pokemon['Ataque_Especial'], fila_pokemon['Defensa_Especial'], fila_pokemon['Velocidad'])
      pokemon.lista_movimientos = lista_movimientos
      return pokemon

  def obtener_efectividad(self, tipo_atacante, tipo_defensor):
    with self.crear_conexion() as conexion:
      cursor = conexion.cursor()
      cursor.execute("SELECT Multiplicador FROM Efectividad_Tipos WHERE ID_Tipo_Atacante = ? AND ID_Tipo_Defensor = ?", (tipo_atacante, tipo_defensor))
      fila = cursor.fetchone()
      return fila['Multiplicador'] if fila else 1.0

  def registrar_batalla(self, nombre_j1, nombre_j2, nombre_ganador, turnos):
    try:
      with self.crear_conexion() as conexion:
        conexion.cursor().execute("INSERT INTO batallas (fecha, jugador_1, jugador_2, ganador, numero_turnos) VALUES (datetime('now'), ?, ?, ?, ?)", (nombre_j1, nombre_j2, nombre_ganador, turnos))
        conexion.commit()
    except: pass

class Movimiento:
  def __init__(self, nombre, tipo_id, categoria, potencia, precision, pp_maximos):
    self.nombre = nombre
    self.tipo_id = tipo_id
    self.categoria = categoria
    self.potencia = potencia
    self.precision = precision
    self.pp_maximos = pp_maximos
    self.pp_actuales = pp_maximos
    self.lista_efectos = []

  def es_valido(self):
    return self.pp_actuales > 0

class Pokemon:
  def __init__(self, id_pokemon, nombre, tipo_1, tipo_2, salud_maxima, ataque, defensa, ataque_especial, defensa_especial, velocidad):
    self.id = id_pokemon
    self.nombre = nombre
    self.tipo_1 = tipo_1
    self.tipo_2 = tipo_2
    self.salud_maxima = salud_maxima
    self.salud_actual = salud_maxima
    self.ataque = ataque
    self.defensa = defensa
    self.ataque_especial = ataque_especial
    self.defensa_especial = defensa_especial
    self.velocidad = velocidad
    self.lista_movimientos = []
    self.estado_persistente = None
    self.turnos_estado = 0
    self.estado_volatil = None
    self.etapas_estadisticas = {'Ataque': 0, 'Defensa': 0, 'Ataque_Especial': 0, 'Defensa_Especial': 0, 'Velocidad': 0}

  def esta_debilitado(self):
    return self.salud_actual <= 0

  def obtener_estadistica(self, nombre_estadistica):
    valor_base = getattr(self, nombre_estadistica)
    if nombre_estadistica == 'velocidad' and self.estado_persistente == 'paralisis': valor_base //= 2
    etapa = self.etapas_estadisticas.get(nombre_estadistica.title(), 0)
    return max(1, int(valor_base * ((2 + etapa) / 2 if etapa >= 0 else 2 / (2 - etapa))))

class Entrenador:
  def __init__(self, nombre):
    self.nombre = nombre
    self.equipo = []
    self.indice_pokemon_activo = 0

  def obtener_pokemon_activo(self):
    return self.equipo[self.indice_pokemon_activo]

  def tiene_pokemon_disponibles(self):
    return any(not pokemon.esta_debilitado() for pokemon in self.equipo)

class Batalla:
  def __init__(self, jugador_1, jugador_2, repositorio):
    self.jugador_1 = jugador_1
    self.jugador_2 = jugador_2
    self.repositorio = repositorio
    self.turno_actual = 1
    self.historial_combate = []
    self.batalla_terminada = False
    self.accion_jugador_1 = None
    self.accion_jugador_2 = None

  def registrar_evento(self, mensaje):
    self.historial_combate.append(mensaje)
    if len(self.historial_combate) > 12: self.historial_combate.pop(0)

  def calcular_daño(self, pokemon_atacante, pokemon_defensor, movimiento):
    if movimiento.categoria == 'Estado' or movimiento.potencia == 0: return 0
    if random.randint(1, 100) > movimiento.precision: self.registrar_evento(f"⚠️ ¡{pokemon_atacante.nombre} falló!"); return 0
    ataque = pokemon_atacante.obtener_estadistica('ataque' if movimiento.categoria == 'Fisico' else 'ataque_especial')
    defensa = pokemon_defensor.obtener_estadistica('defensa' if movimiento.categoria == 'Fisico' else 'defensa_especial')
    efectividad = self.repositorio.obtener_efectividad(movimiento.tipo_id, pokemon_defensor.tipo_1)
    if pokemon_defensor.tipo_2: efectividad *= self.repositorio.obtener_efectividad(movimiento.tipo_id, pokemon_defensor.tipo_2)
    if efectividad == 0: self.registrar_evento(f"❌ No afecta a {pokemon_defensor.nombre}..."); return 0
    bonificacion_mismo_tipo = 1.5 if movimiento.tipo_id in (pokemon_atacante.tipo_1, pokemon_atacante.tipo_2) else 1.0
    multiplicador_critico = 1.5 if random.random() < 0.0625 else 1.0
    factor_aleatorio = random.uniform(0.85, 1.0)
    daño_final = int(((((2 * 50 / 5 + 2) * movimiento.potencia * ataque / max(1, defensa)) / 50) + 2) * bonificacion_mismo_tipo * efectividad * multiplicador_critico * factor_aleatorio)
    if multiplicador_critico > 1: self.registrar_evento("💥 ¡Golpe crítico!")
    if efectividad > 1: self.registrar_evento("🔥 ¡Súper efectivo!")
    self.registrar_evento(f"💢 Hizo {daño_final} de daño.")
    return daño_final

  def ejecutar_movimiento(self, pokemon_atacante, pokemon_defensor, movimiento):
    movimiento.pp_actuales -= 1
    self.registrar_evento(f"🔹 ¡{pokemon_atacante.nombre} usó {movimiento.nombre}!")
    if pokemon_defensor.estado_volatil == 'protegido' and movimiento.categoria != 'Estado': self.registrar_evento(f"🛡️ {pokemon_defensor.nombre} se protegió!"); return
    daño = self.calcular_daño(pokemon_atacante, pokemon_defensor, movimiento)
    if daño > 0: pokemon_defensor.salud_actual = max(0, pokemon_defensor.salud_actual - daño)

    nombre_movimiento = movimiento.nombre.lower().replace('ó', 'o').replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ú', 'u')
    if any(x in nombre_movimiento for x in ['amortiguador', 'recuperacion', 'truco', 'sintesis', 'descanso']):
      if 'descanso' in nombre_movimiento: pokemon_atacante.salud_actual = pokemon_atacante.salud_maxima; pokemon_atacante.estado_persistente = 'dormido'; pokemon_atacante.turnos_estado = 2; self.registrar_evento(f"💚 {pokemon_atacante.nombre} curó toda su vida pero se durmió.")
      else: pokemon_atacante.salud_actual = min(pokemon_atacante.salud_maxima, pokemon_atacante.salud_actual + pokemon_atacante.salud_maxima // 2); self.registrar_evento(f"💚 {pokemon_atacante.nombre} curó PS.")
    elif any(x in nombre_movimiento for x in ['gigadrenado', 'drenaje', 'drenadoras']):
      if daño > 0: curacion = max(1, daño // 2); pokemon_atacante.salud_actual = min(pokemon_atacante.salud_maxima, pokemon_atacante.salud_actual + curacion); self.registrar_evento(f"💚 {pokemon_atacante.nombre} drenó {curacion} PS.")
    elif 'danza espada' in nombre_movimiento: pokemon_atacante.etapas_estadisticas['Ataque'] = min(6, pokemon_atacante.etapas_estadisticas['Ataque'] + 2); self.registrar_evento(f"⚔️ ¡Ataque de {pokemon_atacante.nombre} subió!")
    elif 'agilidad' in nombre_movimiento: pokemon_atacante.etapas_estadisticas['Velocidad'] = min(6, pokemon_atacante.etapas_estadisticas['Velocidad'] + 2); self.registrar_evento(f"💨 ¡Velocidad de {pokemon_atacante.nombre} subió!")
    elif 'amnesia' in nombre_movimiento: pokemon_atacante.etapas_estadisticas['Defensa_Especial'] = min(6, pokemon_atacante.etapas_estadisticas['Defensa_Especial'] + 2); self.registrar_evento(f"🧠 ¡Defensa Esp. de {pokemon_atacante.nombre} subió!")
    elif 'paz mental' in nombre_movimiento: pokemon_atacante.etapas_estadisticas['Ataque_Especial'] = min(6, pokemon_atacante.etapas_estadisticas['Ataque_Especial'] + 1); pokemon_atacante.etapas_estadisticas['Defensa_Especial'] = min(6, pokemon_atacante.etapas_estadisticas['Defensa_Especial'] + 1); self.registrar_evento(f"🧘 ¡Stats Esp. de {pokemon_atacante.nombre} subieron!")
    elif 'maquinac' in nombre_movimiento: pokemon_atacante.etapas_estadisticas['Ataque_Especial'] = min(6, pokemon_atacante.etapas_estadisticas['Ataque_Especial'] + 2); self.registrar_evento(f"⚙️ ¡Atq. Esp. de {pokemon_atacante.nombre} subió mucho!")
    elif 'danza dragon' in nombre_movimiento: pokemon_atacante.etapas_estadisticas['Ataque'] = min(6, pokemon_atacante.etapas_estadisticas['Ataque'] + 1); pokemon_atacante.etapas_estadisticas['Velocidad'] = min(6, pokemon_atacante.etapas_estadisticas['Velocidad'] + 1); self.registrar_evento(f"🐉 ¡Ataque y Velocidad de {pokemon_atacante.nombre} subieron!")
    elif 'grunido' in nombre_movimiento: pokemon_defensor.etapas_estadisticas['Ataque'] = max(-6, pokemon_defensor.etapas_estadisticas['Ataque'] - 1); self.registrar_evento(f"📉 ¡Ataque de {pokemon_defensor.nombre} bajó!")
    elif 'latigo' in nombre_movimiento: pokemon_defensor.etapas_estadisticas['Defensa'] = max(-6, pokemon_defensor.etapas_estadisticas['Defensa'] - 1); self.registrar_evento(f"📉 ¡Defensa de {pokemon_defensor.nombre} bajó!")
    elif 'rayo burbuja' in nombre_movimiento: pokemon_defensor.etapas_estadisticas['Velocidad'] = max(-6, pokemon_defensor.etapas_estadisticas['Velocidad'] - 1); self.registrar_evento(f"📉 ¡Velocidad de {pokemon_defensor.nombre} bajó!")
    elif 'protec' in nombre_movimiento or 'detec' in nombre_movimiento: pokemon_atacante.estado_volatil = 'protegido'; self.registrar_evento(f"🛡️ {pokemon_atacante.nombre} se protegió.")
    elif 'tox' in nombre_movimiento and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'toxico'; self.registrar_evento(f"☠️ {pokemon_defensor.nombre} envenenado!")
    elif 'veneno' in nombre_movimiento and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'veneno'; self.registrar_evento(f"☠️ {pokemon_defensor.nombre} envenenado!")
    elif 'quem' in nombre_movimiento and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'quemadura'; self.registrar_evento(f"🔥 {pokemon_defensor.nombre} quemado!")
    elif 'paral' in nombre_movimiento and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'paralisis'; self.registrar_evento(f"⚡ {pokemon_defensor.nombre} paralizado!")
    elif any(x in nombre_movimiento for x in ['dorm', 'hipno', 'bostezo']) and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'dormido'; pokemon_defensor.turnos_estado = 3; self.registrar_evento(f"💤 {pokemon_defensor.nombre} se durmió!")
    elif 'confus' in nombre_movimiento and not pokemon_defensor.estado_volatil: pokemon_defensor.estado_volatil = 'confuso'; self.registrar_evento(f"🌀 {pokemon_defensor.nombre} está confuso!")

    if 'lanzallamas' in nombre_movimiento or 'aliento igneo' in nombre_movimiento or 'fuego fatuo' in nombre_movimiento:
      if random.random() < 0.3 and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'quemadura'; self.registrar_evento(f"🔥 ¡{pokemon_defensor.nombre} se quemó!")
    elif 'rayo' in nombre_movimiento or 'impactrueno' in nombre_movimiento:
      if random.random() < 0.3 and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'paralisis'; self.registrar_evento(f"⚡ ¡{pokemon_defensor.nombre} se paralizó!")
    elif 'punio hielo' in nombre_movimiento or 'rayo hielo' in nombre_movimiento or 'ventisca' in nombre_movimiento:
      if random.random() < 0.3 and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'congelado'; self.registrar_evento(f"❄ ¡{pokemon_defensor.nombre} se congeló!")

    for ef in movimiento.lista_efectos:
      probabilidad = float(ef['Prob']) / 100.0 if ef['Prob'] and float(ef['Prob']) > 1 else float(ef['Prob']) if ef['Prob'] else 1.0
      if random.random() <= probabilidad:
        nombre_efecto = ef['Nombre'].lower().replace('ó', 'o').replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ú', 'u')
        if 'veneno' in nombre_efecto and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'veneno'; self.registrar_evento(f"☠️ ¡{pokemon_defensor.nombre} envenenado!")
        elif 'quem' in nombre_efecto and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'quemadura'; self.registrar_evento(f"🔥 ¡{pokemon_defensor.nombre} quemado!")
        elif 'paral' in nombre_efecto and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'paralisis'; self.registrar_evento(f"⚡ ¡{pokemon_defensor.nombre} paralizado!")
        elif 'congel' in nombre_efecto and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'congelado'; self.registrar_evento(f"❄ ¡{pokemon_defensor.nombre} congelado!")
        elif any(x in nombre_efecto for x in ['dorm', 'sueno']) and not pokemon_defensor.estado_persistente: pokemon_defensor.estado_persistente = 'dormido'; pokemon_defensor.turnos_estado = 3; self.registrar_evento(f"💤 ¡{pokemon_defensor.nombre} dormido!")
        elif 'confus' in nombre_efecto and not pokemon_defensor.estado_volatil: pokemon_defensor.estado_volatil = 'confuso'; self.registrar_evento(f"🌀 ¡{pokemon_defensor.nombre} confuso!")

    if pokemon_defensor.esta_debilitado(): self.registrar_evento(f"💀 ¡{pokemon_defensor.nombre} se debilitó!")

  def aplicar_efectos_fin_turno(self, pokemon):
    if pokemon.esta_debilitado(): return
    if pokemon.estado_persistente == 'veneno': pokemon.salud_actual = max(0, pokemon.salud_actual - pokemon.salud_maxima // 8); self.registrar_evento(f"{pokemon.nombre} sufre veneno.")
    elif pokemon.estado_persistente == 'toxico': pokemon.turnos_estado += 1; pokemon.salud_actual = max(0, pokemon.salud_actual - (pokemon.salud_maxima * pokemon.turnos_estado) // 16); self.registrar_evento(f"{pokemon.nombre} sufre tóxico.")
    elif pokemon.estado_persistente == 'quemadura': pokemon.salud_actual = max(0, pokemon.salud_actual - pokemon.salud_maxima // 16); self.registrar_evento(f"{pokemon.nombre} sufre quemadura.")

  def verificar_ganador(self):
    if not self.jugador_1.tiene_pokemon_disponibles(): self.batalla_terminada = True; self.registrar_evento(f"🏆 ¡{self.jugador_2.nombre} GANA! 🏆"); self.repositorio.registrar_batalla(self.jugador_1.nombre, self.jugador_2.nombre, self.jugador_2.nombre, self.turno_actual)
    elif not self.jugador_2.tiene_pokemon_disponibles(): self.batalla_terminada = True; self.registrar_evento(f"🏆 ¡{self.jugador_1.nombre} GANA! 🏆"); self.repositorio.registrar_batalla(self.jugador_1.nombre, self.jugador_2.nombre, self.jugador_1.nombre, self.turno_actual)

class SimuladorUI:
  def __init__(self):
    self.repositorio = RepositorioPokemon()
    self.todos_los_pokemones = self.repositorio.listar_pokemones()
    self.pantalla_principal = widgets.VBox()
    display(self.pantalla_principal)
    self.reiniciar_juego()

  def reiniciar_juego(self):
    self.jugador_1 = None; self.jugador_2 = None; self.batalla_actual = None
    self.seleccion_equipo_temporal = []; self.seleccion_movimientos_temporal = []; self.indice_pokemon_eligiendo = 0; self.jugador_eligiendo_actualmente = 1; self.equipo_temporal = []
    self.menu_cambio_activo = False; self.cambio_forzado_activo = 0
    if not self.todos_los_pokemones: self.pantalla_principal.children = (widgets.HTML("<h3>Error: La base de datos está vacía.</h3>"),)
    else: self.mostrar_seleccion_equipo(1)

  def obtener_html_sprite(self, id_pokemon, tamano=100):
    for extension in ['.png', '.jpg', '.jpeg']:
      ruta_archivo = f"imagenes/{id_pokemon}{extension}"
      if os.path.exists(ruta_archivo):
        with open(ruta_archivo, 'rb') as archivo: texto_base64 = base64.b64encode(archivo.read()).decode()
        formato = 'jpeg' if extension in ('.jpg', '.jpeg') else 'png'
        return f"<img src='data:image/{formato};base64,{texto_base64}' style='width:{tamano}px;height:{tamano}px;object-fit:contain;'>"
    return f"<div style='width:{tamano}px;height:{tamano}px;background:#ccc;display:flex;align-items:center;justify-content:center;'>#{id_pokemon}</div>"

  def mostrar_seleccion_equipo(self, numero_jugador):
    self.seleccion_equipo_temporal = []
    tarjetas_pokemon = []
    boton_confirmar = widgets.Button(description="Confirmar (0/3)", button_style='primary', disabled=True)
    for datos_pokemon in self.todos_los_pokemones:
      boton_elegir = widgets.Button(description="Elegir", layout={'width': '80px'})
      def al_hacer_clic(boton, poke_data=datos_pokemon):
        if poke_data['ID'] in [p['ID'] for p in self.seleccion_equipo_temporal]:
          self.seleccion_equipo_temporal = [p for p in self.seleccion_equipo_temporal if p['ID'] != poke_data['ID']]; boton.description = "Elegir"; boton.button_style = ''
        elif len(self.seleccion_equipo_temporal) < 3:
          self.seleccion_equipo_temporal.append(poke_data); boton.description = "✓"; boton.button_style = 'success'
        boton_confirmar.disabled = len(self.seleccion_equipo_temporal) == 0
        boton_confirmar.description = f"Confirmar ({len(self.seleccion_equipo_temporal)}/3)"
      boton_elegir.on_click(al_hacer_clic)
      tarjetas_pokemon.append(widgets.VBox([widgets.HTML(self.obtener_html_sprite(datos_pokemon['ID'], 80)), widgets.HTML(f"<div style='text-align:center'>{datos_pokemon['Nombre']}</div>"), boton_elegir]))
    boton_confirmar.on_click(lambda x: self.iniciar_seleccion_movimientos(numero_jugador))
    self.pantalla_principal.children = (widgets.VBox([widgets.HTML(f"<h3>Jugador {numero_jugador}: Selecciona tu Equipo</h3>"), widgets.GridBox(tarjetas_pokemon, layout={'grid_template_columns': 'repeat(6, 100px)', 'grid_gap': '10px'}), boton_confirmar]),)

  def iniciar_seleccion_movimientos(self, numero_jugador):
    self.jugador_eligiendo_actualmente = numero_jugador; self.indice_pokemon_eligiendo = 0; self.equipo_temporal = []
    for datos_pokemon in self.seleccion_equipo_temporal:
      objeto_pokemon = self.repositorio.cargar_pokemon(datos_pokemon['ID']); objeto_pokemon.lista_movimientos = []; self.equipo_temporal.append(objeto_pokemon)
    self.mostrar_seleccion_movimientos()

  def mostrar_seleccion_movimientos(self):
    pokemon_actual = self.equipo_temporal[self.indice_pokemon_eligiendo]
    movimientos_disponibles = self.repositorio.cargar_pokemon(pokemon_actual.id).lista_movimientos
    self.seleccion_movimientos_temporal = []
    botones_movimientos = []
    boton_confirmar = widgets.Button(description="Confirmar Movs (0/4)", button_style='primary', disabled=True)
    for movimiento in movimientos_disponibles:
      boton_movimiento = widgets.Button(description=f"{movimiento.nombre} (Pot: {movimiento.potencia})", layout={'width': '250px'})
      def al_hacer_clic_movimiento(boton, mov=movimiento):
        if mov in self.seleccion_movimientos_temporal: self.seleccion_movimientos_temporal.remove(mov); boton.button_style = ''
        elif len(self.seleccion_movimientos_temporal) < 4: self.seleccion_movimientos_temporal.append(mov); boton.button_style = 'success'
        boton_confirmar.disabled = len(self.seleccion_movimientos_temporal) == 0
        boton_confirmar.description = f"Confirmar Movs ({len(self.seleccion_movimientos_temporal)}/4)"
      boton_movimiento.on_click(al_hacer_clic_movimiento); botones_movimientos.append(boton_movimiento)
    if not botones_movimientos: botones_movimientos.append(widgets.HTML("<p style='color:red;'>⚠️ Este Pokémon no tiene movimientos en la base de datos.</p>"))
    def confirmar_movimientos(boton):
      self.equipo_temporal[self.indice_pokemon_eligiendo].lista_movimientos = self.seleccion_movimientos_temporal
      self.indice_pokemon_eligiendo += 1
      if self.indice_pokemon_eligiendo < len(self.equipo_temporal): self.mostrar_seleccion_movimientos()
      elif self.jugador_eligiendo_actualmente == 1: self.jugador_1 = Entrenador("Jugador 1"); self.jugador_1.equipo = self.equipo_temporal; self.mostrar_seleccion_equipo(2)
      else: self.jugador_2 = Entrenador("Jugador 2"); self.jugador_2.equipo = self.equipo_temporal; self.iniciar_batalla()
    boton_confirmar.on_click(confirmar_movimientos)
    self.pantalla_principal.children = (widgets.VBox([widgets.HTML(f"<h3>Jugador {self.jugador_eligiendo_actualmente}: Movs para {pokemon_actual.nombre}</h3>"), widgets.HBox([widgets.HTML(self.obtener_html_sprite(pokemon_actual.id, 120)), widgets.VBox(botones_movimientos)]), boton_confirmar]),)

  def iniciar_batalla(self):
    self.batalla_actual = Batalla(self.jugador_1, self.jugador_2, self.repositorio)
    self.batalla_actual.registrar_evento("⚔️ ¡Inicio de Batalla! ⚔️")
    self.jugador_eligiendo_actualmente = 1; self.menu_cambio_activo = False; self.cambio_forzado_activo = 0; self.mostrar_batalla()

  def mostrar_batalla(self):
    batalla = self.batalla_actual; poke_j1 = batalla.jugador_1.obtener_pokemon_activo(); poke_j2 = batalla.jugador_2.obtener_pokemon_activo()
    log_html = "<br>".join([f"<div style='color:{'red' if 'daño' in l or 'debilitó' in l else 'green' if 'curó' in l else 'white'}'>{l}</div>" for l in batalla.historial_combate])
    ui_superior = widgets.HBox([widgets.HTML(self.obtener_html_sprite(poke_j2.id, 120)), widgets.HTML(f"<b>{poke_j2.nombre}</b><br>HP: {poke_j2.salud_actual}/{poke_j2.salud_maxima}<br>Est: {poke_j2.estado_persistente or poke_j2.estado_volatil or 'Normal'}")], layout={'justify_content': 'flex-end'})
    ui_inferior = widgets.HBox([widgets.HTML(f"<b>{poke_j1.nombre}</b><br>HP: {poke_j1.salud_actual}/{poke_j1.salud_maxima}<br>Est: {poke_j1.estado_persistente or poke_j1.estado_volatil or 'Normal'}"), widgets.HTML(self.obtener_html_sprite(poke_j1.id, 120))])
    ui_historial = widgets.HTML(f"<div style='height:150px; overflow-y:auto; background:#333; color:white; padding:10px; border-radius:5px;'>{log_html}</div>")
    panel_acciones = widgets.VBox()

    if batalla.batalla_terminada:
      boton_reiniciar = widgets.Button(description="Reiniciar Juego", button_style='danger'); boton_reiniciar.on_click(lambda x: self.reiniciar_juego()); panel_acciones.children = (boton_reiniciar,)
    elif self.cambio_forzado_activo == 1: panel_acciones.children = (self.mostrar_cambio_forzado(batalla.jugador_1, 1),)
    elif self.cambio_forzado_activo == 2: panel_acciones.children = (self.mostrar_cambio_forzado(batalla.jugador_2, 2),)
    elif batalla.accion_jugador_1 and batalla.accion_jugador_2: self.resolver_turno(); return
    else:
      if self.menu_cambio_activo: panel_acciones.children = (self.mostrar_menu_cambio(batalla.jugador_1 if self.jugador_eligiendo_actualmente == 1 else batalla.jugador_2, self.jugador_eligiendo_actualmente),)
      else: panel_acciones.children = (self.mostrar_menu_acciones(batalla.jugador_1 if self.jugador_eligiendo_actualmente == 1 else batalla.jugador_2, self.jugador_eligiendo_actualmente),)

    self.pantalla_principal.children = (widgets.VBox([widgets.HTML(f"<h3 style='text-align:center'>TURNO {batalla.turno_actual}</h3>"), ui_superior, ui_inferior, ui_historial, panel_acciones]),)

  def mostrar_menu_acciones(self, entrenador, numero_jugador):
    botones_movimientos = []
    for movimiento in entrenador.obtener_pokemon_activo().lista_movimientos:
      boton = widgets.Button(description=f"{movimiento.nombre} ({movimiento.pp_actuales}/{movimiento.pp_maximos})", disabled=not movimiento.es_valido(), button_style='danger' if movimiento.categoria == 'Fisico' else 'info' if movimiento.categoria == 'Especial' else 'success', layout={'width': '180px', 'margin': '2px'})
      boton.on_click(lambda x, mov=movimiento: self.elegir_movimiento(mov, numero_jugador)); botones_movimientos.append(boton)
    boton_cambiar = widgets.Button(description="Cambiar Pokémon", button_style='warning', layout={'width': '180px', 'margin': '2px'}); boton_cambiar.on_click(lambda x: self.abrir_menu_cambio())
    return widgets.VBox([widgets.HTML(f"<h4>Jugador {numero_jugador}: Elige Acción</h4>"), widgets.HBox(botones_movimientos, layout={'flex_flow': 'row wrap'}), boton_cambiar])

  def mostrar_menu_cambio(self, entrenador, numero_jugador):
    botones_cambio = []
    for indice, pokemon in enumerate(entrenador.equipo):
      if indice == entrenador.indice_pokemon_activo or pokemon.esta_debilitado(): continue
      boton = widgets.Button(description=f"Ir a {pokemon.nombre} (HP:{pokemon.salud_actual})", layout={'width': '200px', 'margin': '2px'})
      boton.on_click(lambda x, idx=indice: self.forzar_cambio(entrenador, idx, numero_jugador, False)); botones_cambio.append(boton)
    boton_volver = widgets.Button(description="⬅️ Volver", button_style='primary', layout={'width': '200px', 'margin': '2px'}); boton_volver.on_click(lambda x: self.volver_al_menu()); botones_cambio.append(boton_volver)
    return widgets.VBox([widgets.HTML(f"<h4>Jugador {numero_jugador}: Elige Relevo</h4>"), widgets.VBox(botones_cambio)])

  def mostrar_cambio_forzado(self, entrenador, numero_jugador):
    botones_cambio = []
    for indice, pokemon in enumerate(entrenador.equipo):
      if indice == entrenador.indice_pokemon_activo or pokemon.esta_debilitado(): continue
      boton = widgets.Button(description=f"Ir a {pokemon.nombre} (HP:{pokemon.salud_actual})", layout={'width': '200px', 'margin': '2px'})
      boton.on_click(lambda x, idx=indice: self.forzar_cambio(entrenador, idx, numero_jugador, True)); botones_cambio.append(boton)
    return widgets.VBox([widgets.HTML(f"<h4>Jugador {numero_jugador}: Elige Relevo</h4>"), widgets.VBox(botones_cambio)])

  def abrir_menu_cambio(self): self.menu_cambio_activo = True; self.mostrar_batalla()
  def volver_al_menu(self): self.menu_cambio_activo = False; self.mostrar_batalla()

  def forzar_cambio(self, entrenador, indice_nuevo, numero_jugador, es_forzado):
    entrenador.indice_pokemon_activo = indice_nuevo; self.menu_cambio_activo = False; self.cambio_forzado_activo = 0
    if not es_forzado: self.elegir_cambio(("CAMBIO", indice_nuevo), numero_jugador)
    else:
      if self.batalla_actual.jugador_1.obtener_pokemon_activo().esta_debilitado(): self.cambio_forzado_activo = 1
      elif self.batalla_actual.jugador_2.obtener_pokemon_activo().esta_debilitado(): self.cambio_forzado_activo = 2
      else: self.batalla_actual.turno_actual += 1; self.batalla_actual.accion_jugador_1 = None; self.batalla_actual.accion_jugador_2 = None; self.jugador_eligiendo_actualmente = 1
      self.mostrar_batalla()

  def elegir_movimiento(self, movimiento, numero_jugador):
    self.menu_cambio_activo = False
    if numero_jugador == 1: self.batalla_actual.accion_jugador_1 = movimiento; self.jugador_eligiendo_actualmente = 2
    else: self.batalla_actual.accion_jugador_2 = movimiento
    self.mostrar_batalla()

  def elegir_cambio(self, accion_cambio, numero_jugador):
    self.menu_cambio_activo = False
    if numero_jugador == 1: self.batalla_actual.accion_jugador_1 = accion_cambio; self.jugador_eligiendo_actualmente = 2
    else: self.batalla_actual.accion_jugador_2 = accion_cambio
    self.mostrar_batalla()

  def resolver_turno(self):
    batalla = self.batalla_actual; batalla.registrar_evento(f"<br><b>--- Turno {batalla.turno_actual} ---</b>")
    j1_cambia = isinstance(batalla.accion_jugador_1, tuple); j2_cambia = isinstance(batalla.accion_jugador_2, tuple)
    if j1_cambia: batalla.jugador_1.indice_pokemon_activo = batalla.accion_jugador_1[1]; batalla.registrar_evento("🔄 J1 cambió.")
    if j2_cambia: batalla.jugador_2.indice_pokemon_activo = batalla.accion_jugador_2[1]; batalla.registrar_evento("🔄 J2 cambió.")
    if not j1_cambia and not j2_cambia:
      velocidad_j1 = batalla.jugador_1.obtener_pokemon_activo().obtener_estadistica('velocidad'); velocidad_j2 = batalla.jugador_2.obtener_pokemon_activo().obtener_estadistica('velocidad')
      orden_ataque = [1, 2] if velocidad_j1 > velocidad_j2 or (velocidad_j1 == velocidad_j2 and random.random() < 0.5) else [2, 1]
      for turno in orden_ataque:
        if batalla.batalla_terminada: break
        if turno == 1: batalla.ejecutar_movimiento(batalla.jugador_1.obtener_pokemon_activo(), batalla.jugador_2.obtener_pokemon_activo(), batalla.accion_jugador_1)
        else: batalla.ejecutar_movimiento(batalla.jugador_2.obtener_pokemon_activo(), batalla.jugador_1.obtener_pokemon_activo(), batalla.accion_jugador_2)
    batalla.aplicar_efectos_fin_turno(batalla.jugador_1.obtener_pokemon_activo()); batalla.aplicar_efectos_fin_turno(batalla.jugador_2.obtener_pokemon_activo())
    for poke in [batalla.jugador_1.obtener_pokemon_activo(), batalla.jugador_2.obtener_pokemon_activo()]:
      if poke.estado_volatil == 'protegido': poke.estado_volatil = None
    batalla.verificar_ganador()
    if batalla.batalla_terminada: self.mostrar_batalla(); return
    if batalla.jugador_1.obtener_pokemon_activo().esta_debilitado(): self.cambio_forzado_activo = 1
    elif batalla.jugador_2.obtener_pokemon_activo().esta_debilitado(): self.cambio_forzado_activo = 2
    else: batalla.turno_actual += 1; batalla.accion_jugador_1 = None; batalla.accion_jugador_2 = None; self.jugador_eligiendo_actualmente = 1
    self.mostrar_batalla()

simulador = SimuladorUI()

VBox()